In [2]:
import cobra
import modelseedpy
import cobrakbase
import pandas as pd
from cobrakbase.kbaseapi_cache import KBaseCache

modelseedpy 0.3.3
cobrakbase 0.3.1


In [45]:
rast = modelseedpy.RastClient()
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache')

In [46]:
ws_id = 155805
ws_os = kbase.list_objects(ws_id)

In [57]:
import os
genomes = {}
for o in ws_os:
    if o[2].startswith('KBaseGenomes.Genome'):
        genome = kbase.get_from_ws(o[1], ws_id)
        genomes[genome.info.id] = genome
        #for f in genome.features:
        #    f.ontology_terms = {}
        #rast.annotate_genome(genome)

In [4]:
df_metadata = pd.read_csv('metadata.tsv', sep='\t')

In [64]:
templates_fetch = {
    'A': 'ArchaeaModelTemplateV5_test',
    'N': 'GramNegModelTemplateV5_test',
    'P': 'GramPosModelTemplateV5_test',
}
templates = {}
for c, template_id in templates_fetch.items():
    templates[c] = kbase.get_from_ws(template_id, 12218)

In [71]:
from modelseedpy import MSBuilder
models = {}
for row_id, d in df_metadata.iterrows():
    genome_id = d['genome_id']
    genome_class = d['ms_knn_ACNP_RAST_filter_01_17_2023']
    genome = genomes[genome_id]
    b = MSBuilder(genome, templates[genome_class], genome_id)
    models[genome_id] = b.build(genome_id, annotate_with_rast=False)

In [75]:
import pandas as pd
import math
from modelseedpy.core.msmedia import MSMedia
from modelseedpy_ext.utils import load_medias
def load_medias(filep, index_col=0):
    medias = pd.read_csv(filep, sep='\t', index_col=index_col).to_dict()
    return medias
medias = load_medias('/opt/python/ModelSEEDpy/modelseedpy/data/atp_medias.tsv', 0)
media_data = {}
for media_id in medias:
    d = medias[media_id]
    media_const = {
        'cpd00001': 1000, 
        'cpd00067': 1000
    }
    for cpd_id in d:
        v = d[cpd_id]
        if v != 0 and not math.isnan(v):
            media_const[cpd_id[3:-3]] = v
    media = MSMedia.from_dict(media_const)
    media.id = media_id
    media_data[media_id] = media
"""
media_s_matrix = MSMedia.from_dict({
    'cpd00001': 1000, 
    'cpd00067': 1000, 
    'cpd00048': 1,
    'cpdETCMe': 1000,
})
media_s_matrix.id = 'Sulfate.Matrix'
media_data[media_s_matrix.id] = media_s_matrix
"""
pass

In [76]:
for m in models:
    model = models[m]
    cobra.io.write_sbml_model(model, f'/scratch/fliu/data/cliff/models/base/{m}.xml')
    print(m, len(model.reactions))

Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.8.contigs__.RAST 1151
Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.20.contigs__.RAST 1261
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.4.contigs__.RAST 1106
Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.46.contigs__.RAST 1121
Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_concoct_out.21.contigs__.RAST 645
Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_metabat.31.contigs__.RAST 780
Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct_out.29.contigs__.RAST 785
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_metabat.50.contigs__.RAST 1155
Salt_Pond_MetaG_R1_B_D2_MG_DASTool_bins_metabat.18.contigs__.RAST 1196
Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.32.contigs__.RAST 1098
Salt_Pond_MetaG_R1_A_D1_MG_DASTool_bins.metabat.15.contigs__.RAST 1058
Salt_Pond_MetaG_R2_C_D1_MG_DASTool_bins_concoct_out.88.contigs__.RAST 945
Salt_Pond_MetaG_R2_restored_C_black_MG_DASTool_bins_concoct_out.57.contigs__.RAST 989
Salt_Pond_MetaG_R2_A_D1_MG_DAST

In [77]:
template_core_bac = kbase.get_from_ws('CoreBacV5.1', 12218)
template_core = {
    'A': template_core_bac,
    'N': template_core_bac,
    'P': template_core_bac
}

/opt/env/python3_modelseed/lib/python3.11/site-packages/urllib3/util/ssl_.py:262: DeprecationWarning: ssl.PROTOCOL_TLS is deprecated
  context = SSLContext(ssl_version or ssl.PROTOCOL_SSLv23)
/opt/env/python3_modelseed/lib/python3.11/site-packages/urllib3/connection.py:374: DeprecationWarning: ssl.match_hostname() is deprecated
  match_hostname(cert, asserted_hostname)


In [ ]:
import json
from modelseedpy.core.msatpcorrection import min_gap, MSATPCorrection
errors = {}
for row_id, d in df_metadata.iterrows():
    try:
        genome_id = d['genome_id']
        model = cobra.io.read_sbml_model(f'/scratch/fliu/data/cliff/models/base/{genome_id}.xml')
        _template_core = template_core_bac
        atpcorrection = MSATPCorrection(model,
                                        _template_core,
                                        [],
                                        load_default_medias=False,
                                        max_gapfilling=10,
                                        gapfilling_delta=0,
                                        atp_hydrolysis_id='rxn00062_c0',
                                        forced_media=[])
        for media_id, media in media_data.items():
            media.name = media_id
            min_obj = min_gap.get(media_id, 0.01)
            atpcorrection.atp_medias.append([media, min_obj])
        atpcorrection.evaluate_growth_media()
        atpcorrection.determine_growth_media()
        gapfill_stats = {}
        for media, stats in atpcorrection.media_gapfill_stats.items():
            if stats:
                gapfill_stats[media.id] = stats
                if 'media' in gapfill_stats[media.id]:
                    del gapfill_stats[media.id]['media']
        with open(f'/scratch/fliu/data/cliff/models/core_gapfill/{genome_id}.json', 'w') as fh:
            fh.write(json.dumps(gapfill_stats, indent=2))
        
    except Exception as e:
        errors[genome_id] = e

ERROR:cobra.io.sbml:'' is not a valid SBML 'SId'.
INFO:modelseedpy.core.msmodelutl:cpd00075 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00209 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00418 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00075 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00209 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00418 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00075 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00209 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00418 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd08021 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00811 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd08021 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd00811 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd11640 not found in model!
INFO:modelseedpy.core.msmodelutl:cpd11640 not found in model!
INFO:modelseedpy.cor

In [ ]:
1

In [2]:
kbase = cobrakbase.KBaseAPI()

In [3]:
RAST_test_beta_combined_app = kbase.get_from_ws('RAST_test_beta_combined_app', 156555)
RAST_test_beta = kbase.get_from_ws('RAST_test_beta', 156555)
RAST_test2 = kbase.get_from_ws('RAST_test2', 156555)
E_coli_RAST = kbase.get_from_ws('E_coli_RAST', 156555)
E_coli_original = kbase.get_from_ws('E_coli_str._K-12_substr._MG1655', 156555)

In [5]:
genomes = [
    RAST_test_beta_combined_app,
    RAST_test_beta,
    RAST_test2,
    E_coli_RAST,
    E_coli_original
]

In [14]:
ids = {g.id for g in g.features}
for g in genomes:
    print(g.info.id, len(g.features))
    ids &= {g.id for g in g.features}
len(ids)

RAST_test_beta_combined_app 4298
RAST_test_beta 4355
RAST_test2 4298
E_coli_RAST 4298
E_coli_str._K-12_substr._MG1655 4355


4298

In [23]:
for g in genomes:
    missing =  {g.id for g in g.features} - ids
    for f_id in missing:
        f = g.features.get_by_id(f_id)
        print(g.info.id, len(f.seq), f.ontology_terms)
        break
    break

RAST_test_beta 232 {'RAST': ['Putative transferase'], 'SSO': ['SSO:000023648']}
E_coli_str._K-12_substr._MG1655 232 {'RAST': ['putative acyltransferase, N-terminal fragment']}


In [31]:
g.features.b0001.cdss

['b0001_CDS_1']

In [40]:
cds = g.cdss[0]

In [41]:
cds.id

In [36]:
{x.id for x in g.cdss}

{<cobrakbase.core.kbasegenome.genome_cds_feature.KBaseGenomeCDS at 0x7f73ddf60050>,
 ...}

In [8]:
import json
class SvgGridPlot:
    
    def __init__(self):
        self.grid = {}
        self.x = set()
        self.y = set()
    
    def add(self, x, y, v):
        self.x.add(x)
        self.y.add(y)
        if x not in self.grid:
            self.grid[x] = {}
        self.grid[x][y] = v
grid_plot = SvgGridPlot()
for row_id, d in df_metadata.iterrows():
    try:
        genome_id = d['genome_id']
        with open(f'/home/fliu/scratch/data/cliff/models/core_gapfill/{genome_id}.json', 'r') as fh:
            atp_data = json.load(fh)
            for m in atp_data:
                score = get_gap_score(atp_data[m])
                grid_plot.add(genome_id, m, score)
    except Exception as e:
        print(e)

In [7]:
def get_gap_score(t):
    score = 0
    g_rev = t['reversed']
    g_new = t['new']
    if len(g_rev) > 1:
        print('!')
    for k in g_new:
        if not k.startswith('EX_'):
            score += 1
    return score

for m in atp_data:
    score = get_gap_score(atp_data[m])
    print(m, score)

Glc.O2 0
Ac.O2 0
Etho.O2 0
Pyr.O2 0
Glyc.O2 1
Fum.O2 1
Succ.O2 2
Akg.O2 1
LLac.O2 1
Dlac.O2 1
For.O2 0
Glc 0
Ac 10
Pyr 0
Glyc 2
Fum 1
Succ 2
Akg 2
Llac 11
Dlac 3
For 13
For.NO2 0
For.NO3 10
For.NO 10
Pyr.NO2 0
Pyr.NO3 0
Pyr.NO 10
Ac.NO2 0
Ac.NO3 0
Ac.NO 0
Glc.DMSO 0
Glc.TMAO 0
Pyr.DMSO 0
Pyr.TMAO 0
Pyr.SO4 4
Pyr.SO3 0
H2.CO2 12
H2.Ac 12
For.SO4.H2 4
LLac.SO4.H2 5
For.SO4 5
LLac.SO4 5
H2.SO4 13
Methanol 7
Methanol.H2 7
Methanamine.H2 7
Dimethylamine.H2 9
Trimethylamine.H2 11


In [161]:
y_order = """
Glc.O2 LLac.O2 Dlac.O2 Fum.O2 Akg.O2 Ac.O2 For.O2 Pyr.O2 Glyc.O2 Succ.O2 Etho.O2 
Glc    Llac    Dlac    Fum    Akg    Ac    For    Pyr    Glyc    Succ
 Ac.NO  Ac.NO2  Ac.NO3 
For.NO For.NO2 For.NO3 
Pyr.NO Pyr.NO2 Pyr.NO3 
Pyr.DMSO Pyr.TMAO Glc.DMSO Glc.TMAO
H2.CO2 H2.Ac Methanamine.H2 Dimethylamine.H2 Trimethylamine.H2 Methanol.H2 Methanol 
Pyr.SO3 Pyr.SO4 For.SO4 H2.SO4 LLac.SO4 For.SO4.H2 LLac.SO4.H2
""".strip().split()
y_order

['Glc.O2',
 'LLac.O2',
 'Dlac.O2',
 'Fum.O2',
 'Akg.O2',
 'Ac.O2',
 'For.O2',
 'Pyr.O2',
 'Glyc.O2',
 'Succ.O2',
 'Etho.O2',
 'Glc',
 'Llac',
 'Dlac',
 'Fum',
 'Akg',
 'Ac',
 'For',
 'Pyr',
 'Glyc',
 'Succ',
 'Ac.NO',
 'Ac.NO2',
 'Ac.NO3',
 'For.NO',
 'For.NO2',
 'For.NO3',
 'Pyr.NO',
 'Pyr.NO2',
 'Pyr.NO3',
 'Pyr.DMSO',
 'Pyr.TMAO',
 'Glc.DMSO',
 'Glc.TMAO',
 'H2.CO2',
 'H2.Ac',
 'Methanamine.H2',
 'Dimethylamine.H2',
 'Trimethylamine.H2',
 'Methanol.H2',
 'Methanol',
 'Pyr.SO3',
 'Pyr.SO4',
 'For.SO4',
 'H2.SO4',
 'LLac.SO4',
 'For.SO4.H2',
 'LLac.SO4.H2']

In [162]:
with open('/scratch/fliu/data/cliff/plots/core/grid_data.json', 'w') as fh:
    fh.write(json.dumps(grid_plot.grid))
with open('/scratch/fliu/data/cliff/plots/core/x_label.json', 'w') as fh:
    fh.write(json.dumps(list(grid_plot.x)))
with open('/scratch/fliu/data/cliff/plots/core/x_label_taxa.json', 'w') as fh:
    fh.write(json.dumps(dict({i:[i] for i in grid_plot.x})))
with open('/scratch/fliu/data/cliff/plots/core/y_label.json', 'w') as fh:
    fh.write(json.dumps(list(y_order)))

In [147]:
for u in grid_plot.y:
    print(u)

For
Ac.O2
LLac.O2
LLac.SO4
Pyr.DMSO
Glc.DMSO
For.O2
Pyr.SO4
Ac.NO2
Glc.O2
Glyc
For.SO4.H2
Succ
Dimethylamine.H2
For.NO
Trimethylamine.H2
Glyc.O2
Succ.O2
Pyr.NO
Pyr.SO3
Methanamine.H2
Pyr
Dlac.O2
Pyr.NO3
Llac
Glc
For.NO3
Methanol
Akg
Fum.O2
Fum
Glc.TMAO
H2.CO2
Etho.O2
H2.SO4
LLac.SO4.H2
Ac
Ac.NO
H2.Ac
For.SO4
Dlac
Pyr.NO2
Pyr.TMAO
Pyr.O2
Ac.NO3
Akg.O2
Methanol.H2
For.NO2


In [40]:
groups = {}
for d in grid_plot.x:
    o = d[len('Salt_Pond_MetaG'):]
    g = None
    l = None
    depth = None
    if o.startswith('SF2'):
        g = 'SF2'
        o = o[4:]
    elif o.startswith('_R2_'):
        g = 'R2'
        o = o[4:]
    elif o.startswith('_R1_'):
        g = 'R1'
        o = o[4:]
    elif o.startswith('_R2A_'):
        g = 'R2A'
        o = o[5:]
    else:
        print('!')
    if '_D1_' in o:
        depth = 'D1'
    elif '_D2_' in o:
        depth = 'D2'
    elif '_H2O_' in o:
        depth = 'H2O'
    if o.startswith('A_'):
        l = 'A'
    elif o.startswith('B_'):
        l = 'B'
    elif o.startswith('C_'):
        l = 'C'
    elif o.startswith('restored'):
        l = 'R'
    #print(g, l, depth, '\t', d)
    if g not in groups:
        groups[g] = {}
    if l not in groups[g]:
        groups[g][l] = {}
    if depth not in groups[g][l]:
        groups[g][l][depth] = set()
    groups[g][l][depth].add(d)
    #print()

In [39]:
for g in groups:
    print(g)
    for l in groups[g]:
        print('\t', l)
        for d in groups[g][l]:
            print('\t\t', d)
            for genome_id in groups[g][l][d]:
                print('\t\t\t', genome_id)

SF2
	 C
		 D2
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.26.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_metabat.19.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_metabat.16.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.3.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.25.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.24.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_metabat.11.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.39.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.13.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.17.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.15.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.8.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool_bins_concoct_out.42.contigs__.RAST
			 Salt_Pond_MetaGSF2_C_D2_MG_DASTool